In [1]:
import sys
import numpy as np

sys.path.append('../../')
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

In [2]:
config = {
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.legacy.Adam(learning_rate=0.001),
}

In [3]:
def get_train_data():
    return np.load('../../data/MNIST/train_data.npy'), np.load('../../data/MNIST/train_labels.npy')

def get_test_data():
    return np.load('../../data/MNIST/test_data.npy'), np.load('../../data/MNIST/test_labels.npy')

def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)
    
    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size:])
            y_train_partitions.append(y_train[i * partition_size:])
        else:
            X_train_partitions.append(X_train[i * partition_size:(i + 1) * partition_size])
            y_train_partitions.append(y_train[i * partition_size:(i + 1) * partition_size])

    return X_train_partitions, y_train_partitions

In [4]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation('softmax'))
    return model

In [5]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config['partitions'])

In [6]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

Provisioner created successfully


In [7]:
rain.test()

divider is running
 divider received: Success recieving the number of workers.
divider is sending data to the coordinator
../../data/X_train_1
 divider received: Success!
../../data/y_train_1
 divider received: Success!
../../data/X_train_2
 divider received: Success!
../../data/y_train_2
 divider received: Success!
../../data/X_train_3
 divider received: Success!
../../data/y_train_3
 divider received: Success!


In [ ]:
# rain.setup_vms()

In [ ]:
# rain.delete_vms()


In [ ]:
# model = rain.train_centralized_sync()

In [ ]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))